# Setup — run this first in your notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Connect to your database
engine = create_engine("sqlite:///amazon_sales.db")  # or postgresql://...
df = pd.read_sql("SELECT * FROM fact_sales f JOIN dim_product p ON f.product_id = p.product_id JOIN dim_date d ON f.date_id = d.date_id JOIN dim_geography g ON f.geo_id = g.geo_id JOIN dim_seller s ON f.seller_id = s.seller_id WHERE f.order_status = 'Delivered'", engine)

# Style config — use this once at the top
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold'
})
sns.set_palette("Blues_d")

# Chart 1 — Monthly revenue trend with MoM annotations

In [ ]:
monthly = df.groupby(['year', 'month', 'month_name'])['sale_amount'].sum().reset_index()
monthly = monthly.sort_values(['year', 'month'])
monthly['mom_growth'] = monthly['sale_amount'].pct_change() * 100
monthly['label'] = monthly['month_name'] + ' ' + monthly['year'].astype(str)

fig = px.line(monthly, x='label', y='sale_amount',
              title='Monthly revenue trend with MoM growth',
              markers=True, template='plotly_white')
fig.update_traces(line_color='#185FA5', line_width=2.5, marker_size=7)

# Annotate MoM growth on each point
for _, row in monthly.iterrows():
    if pd.notna(row['mom_growth']):
        color = '#1D9E75' if row['mom_growth'] >= 0 else '#E24B4A'
        fig.add_annotation(x=row['label'], y=row['sale_amount'],
                           text=f"{row['mom_growth']:+.1f}%",
                           showarrow=False, yshift=14,
                           font=dict(size=10, color=color))

fig.update_layout(xaxis_title='', yaxis_title='Revenue (₹)',
                  yaxis_tickformat=',.0f')
import os
os.makedirs("charts", exist_ok=True)
fig.write_html("charts/01_monthly_revenue.html")
fig.show()

# Chart 2 — Category performance treemap

In [ ]:
cat_data = df.groupby(['category', 'sub_category']).agg(
    revenue=('sale_amount', 'sum'),
    profit=('profit', 'sum'),
    orders=('sale_id', 'count')
).reset_index()
cat_data['margin_pct'] = (cat_data['profit'] / cat_data['revenue'] * 100).round(1)

fig = px.treemap(cat_data,
                 path=['category', 'sub_category'],
                 values='revenue',
                 color='margin_pct',
                 color_continuous_scale='RdYlGn',
                 title='Revenue by category — size = revenue, color = profit margin %',
                 hover_data={'orders': True, 'margin_pct': ':.1f'})
fig.update_layout(template='plotly_white')
fig.write_html("charts/02_category_treemap.html")
fig.show()

# Chart 3 — Profit margin heatmap (category × region)

In [ ]:
pivot = df.groupby(['category', 'region']).apply(
    lambda x: (x['profit'].sum() / x['sale_amount'].sum() * 100)
).round(1).reset_index(name='margin_pct')

heatmap_data = pivot.pivot(index='category', columns='region', values='margin_pct')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Profit margin %'})
ax.set_title('Profit margin % by category and region', pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig("charts/03_margin_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

# Chart 4 — Pareto chart (80/20 rule on SKUs)

In [ ]:
product_rev = df.groupby('product_name')['sale_amount'].sum().sort_values(ascending=False).reset_index()
product_rev['cumulative_pct'] = product_rev['sale_amount'].cumsum() / product_rev['sale_amount'].sum() * 100
product_rev['rank'] = range(1, len(product_rev) + 1)

# Find the 80% cutoff
cutoff_rank = product_rev[product_rev['cumulative_pct'] <= 80].shape[0]

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

bars = ax1.bar(product_rev['rank'], product_rev['sale_amount'],
               color=['#185FA5' if i <= cutoff_rank else '#B5D4F4'
                      for i in product_rev['rank']], width=0.8)
ax2.plot(product_rev['rank'], product_rev['cumulative_pct'],
         color='#E24B4A', linewidth=2, marker='')
ax2.axhline(80, color='#E24B4A', linestyle='--', alpha=0.5, linewidth=1)
ax2.axvline(cutoff_rank, color='gray', linestyle='--', alpha=0.4, linewidth=1)

ax1.set_xlabel('Products ranked by revenue')
ax1.set_ylabel('Revenue (₹)')
ax2.set_ylabel('Cumulative %')
ax2.set_ylim(0, 105)
ax1.set_title(f'Pareto analysis — top {cutoff_rank} products drive 80% of revenue', pad=15)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.savefig("charts/04_pareto_skus.png", dpi=150, bbox_inches='tight')
plt.show()

# Chart 5 — Discount vs profit scatter (the insight chart)

In [ ]:
scatter_data = df.groupby('product_name').agg(
    avg_discount=('discount', 'mean'),
    total_profit=('profit', 'sum'),
    total_revenue=('sale_amount', 'sum'),
    category=('category', 'first')
).reset_index()
scatter_data['avg_discount_pct'] = scatter_data['avg_discount'] * 100

fig = px.scatter(scatter_data, x='avg_discount_pct', y='total_profit',
                 color='category', size='total_revenue',
                 hover_name='product_name',
                 title='Discount % vs total profit by product (size = revenue)',
                 template='plotly_white',
                 labels={'avg_discount_pct': 'Average discount %',
                         'total_profit': 'Total profit (₹)'})
fig.add_vline(x=scatter_data['avg_discount_pct'].mean(),
              line_dash='dash', line_color='gray', annotation_text='Avg discount')
fig.add_hline(y=0, line_color='#E24B4A', line_width=1)
fig.write_html("charts/05_discount_vs_profit.html")
fig.show()

# Chart 6 — Geo choropleth (India state-level)

In [ ]:
state_data = df.groupby('state').agg(
    revenue=('sale_amount', 'sum'),
    orders=('sale_id', 'count')
).reset_index()

# If geojson is unavailable, use a bar chart as fallback:
fig = px.bar(state_data.sort_values('revenue', ascending=True).tail(15),
             x='revenue', y='state', orientation='h',
             title='Top 15 states by revenue', template='plotly_white',
             color='revenue', color_continuous_scale='Blues')
fig.write_html("charts/06_geo_revenue.html")
fig.show()

# Chart 7 — Seller / vendor performance matrix

In [ ]:
vendor = df.groupby('seller_name').agg(
    revenue=('sale_amount', 'sum'),
    orders=('sale_id', 'count'),
    avg_rating=('rating', 'mean'),
    return_rate=('order_status', lambda x: (x == 'Returned').sum() / len(x) * 100)
).reset_index()

fig = px.scatter(vendor, x='return_rate', y='avg_rating',
                 size='revenue', color='revenue',
                 hover_name='seller_name', text='seller_name',
                 title='Vendor quality matrix — return rate vs rating (size = revenue)',
                 color_continuous_scale='Blues', template='plotly_white',
                 labels={'return_rate': 'Return rate %', 'avg_rating': 'Avg rating'})
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_vline(x=vendor['return_rate'].mean(), line_dash='dash', line_color='gray')
fig.add_hline(y=vendor['avg_rating'].mean(), line_dash='dash', line_color='gray')
fig.write_html("charts/07_vendor_matrix.html")
fig.show()

# Save a summary stats table for your README

In [ ]:
summary = {
    'Total Orders': df['sale_id'].nunique(),
    'Total Revenue': f"₹{df['sale_amount'].sum():,.0f}",
    'Total Profit': f"₹{df['profit'].sum():,.0f}",
    'Overall Margin': f"{df['profit'].sum()/df['sale_amount'].sum()*100:.1f}%",
    'Unique Products': df['product_name'].nunique(),
    'Unique Customers': df['customer_id'].nunique(),
    'Avg Order Value': f"₹{df['sale_amount'].mean():,.0f}",
    'Avg Discount': f"{df['discount'].mean()*100:.1f}%"
}
import os
os.makedirs("data", exist_ok=True)
pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])\
  .to_csv("data/summary_stats.csv")
print(pd.DataFrame.from_dict(summary, orient='index', columns=['Value']))